In [0]:
use catalog smart_odoo;
use schema gold;

-- Total Sales  , mom sales growth

CREATE OR REPLACE VIEW gold.vw_monthly_sales_mom AS

WITH monthly_sales AS (

    SELECT
        Cast ( DATE_TRUNC('month', order_date) AS Date) AS month,
        SUM(net_revenue) AS total_sales

    FROM gold.fact_sales

    WHERE order_date IS NOT NULL
      AND order_state IN ('sale', 'done')

    GROUP BY DATE_TRUNC('month', order_date)

)

SELECT
    month,
    total_sales,

    LAG(total_sales) OVER (ORDER BY month) AS prev_month_sales,

    ROUND(
        (
            total_sales -
            LAG(total_sales) OVER (ORDER BY month)
        )
        /
        NULLIF(
            LAG(total_sales) OVER (ORDER BY month),
            0
        ) * 100,
        2
    ) AS mom_growth_percent

FROM monthly_sales;

In [0]:
use catalog smart_odoo;
use schema gold;

-- Total Orders  , mom orders growth

CREATE OR REPLACE VIEW gold.vw_monthly_orders_mom AS

WITH monthly_orders AS (

    SELECT
        Cast ( DATE_TRUNC('month', order_date) AS Date) AS month,
        COUNT(DISTINCT order_id) AS total_orders

    FROM gold.fact_sales

    WHERE order_date IS NOT NULL
      AND order_state IN ('sale', 'done')

    GROUP BY DATE_TRUNC('month', order_date)

)

SELECT
    month,

    total_orders,

    LAG(total_orders) OVER (ORDER BY month) AS prev_month_orders,

    ROUND(
        (
            total_orders -
            LAG(total_orders) OVER (ORDER BY month)
        )
        /
        NULLIF(
            LAG(total_orders) OVER (ORDER BY month),
            0
        ) * 100,
        2
    ) AS mom_growth_percent

FROM monthly_orders;

In [0]:
use catalog smart_odoo;
use schema gold;

-- Avg Order Value  , mom aov growth

CREATE OR REPLACE VIEW gold.vw_monthly_aov_mom AS

WITH monthly_aov AS (

    SELECT
       Cast( DATE_TRUNC('month', order_date) AS date) AS month,

        ROUND(
            SUM(net_revenue)
            /
            NULLIF(COUNT(DISTINCT order_id), 0),
            2
        ) AS avg_order_value

    FROM gold.fact_sales

    WHERE order_date IS NOT NULL
      AND order_state IN ('sale', 'done')

    GROUP BY DATE_TRUNC('month', order_date)

)

SELECT
    month,
    avg_order_value,

    LAG(avg_order_value) OVER (ORDER BY month) AS prev_month_aov,

    ROUND(
        (
            avg_order_value -
            LAG(avg_order_value) OVER (ORDER BY month)
        )
        /
        NULLIF(
            LAG(avg_order_value) OVER (ORDER BY month),
            0
        ) * 100,
        2
    ) AS mom_growth_percent

FROM monthly_aov;

In [0]:
use catalog smart_odoo;
use schema gold;

CREATE OR REPLACE VIEW gold.vw_monthly_customer_kpis AS

WITH customer_first_order AS (

    SELECT
        customer_id,
        MIN(Cast( DATE_TRUNC('month', order_date) AS date)) AS first_order_month

    FROM gold.fact_sales

    WHERE order_date IS NOT NULL
      AND order_state IN ('sale', 'done')

    GROUP BY customer_id

),

monthly_customers AS (

    SELECT
       Cast( DATE_TRUNC('month', fs.order_date) AS date) AS month,

        COUNT(DISTINCT fs.customer_id) AS active_customers

    FROM gold.fact_sales fs

    WHERE fs.order_date IS NOT NULL
      AND fs.order_state IN ('sale', 'done')

    GROUP BY Cast( DATE_TRUNC('month', fs.order_date) AS date)

),

new_customers AS (

    SELECT
        first_order_month AS month,

        COUNT(DISTINCT customer_id) AS new_customers

    FROM customer_first_order

    GROUP BY first_order_month

)

SELECT
    mc.month,

    mc.active_customers,

    COALESCE(nc.new_customers, 0) AS new_customers_this_month

FROM monthly_customers mc

LEFT JOIN new_customers nc
    ON mc.month = nc.month;

In [0]:
CREATE OR REPLACE VIEW gold.vw_sales_revenue_trend AS

WITH monthly_sales AS (

    SELECT
        cast(order_date as date),
        SUM(net_revenue) AS total_revenue

    FROM gold.fact_sales

    WHERE order_state IN ('sale', 'done')

    GROUP BY order_date

)

SELECT
    dd.year_number,
    dd.month_number,
    dd.month_short,

    ROUND(
        COALESCE(SUM(ms.total_revenue), 0),
        2
    ) AS total_revenue

FROM gold.dim_date dd

LEFT JOIN monthly_sales ms
    ON dd.full_date = ms.order_date

WHERE dd.year_number = YEAR(CURRENT_DATE)
  AND dd.month_number <= MONTH(CURRENT_DATE)

GROUP BY
    dd.year_number,
    dd.month_number,
    dd.month_short

ORDER BY
    dd.month_number ASC;

In [0]:
CREATE OR REPLACE VIEW gold.vw_top_products_sales AS

SELECT
    dp.product_id,
    dp.product_name,

    ROUND(
        SUM(fs.net_revenue),
        2
    ) AS total_sales

FROM gold.fact_sales fs

INNER JOIN gold.dim_product dp
    ON fs.product_id = dp.product_id

    WHERE fs.order_state IN ('sale', 'done')

GROUP BY
    dp.product_id,
    dp.product_name;

In [0]:
CREATE OR REPLACE VIEW gold.vw_recent_orders AS

SELECT
    concat('SO' , fs.order_id) AS order_id,

    dc.partner_name As customer_name,

    COUNT(fs.product_qty) AS items_count,

    ROUND(
        SUM(fs.net_revenue),
        2
    ) AS order_amount,

    fs.order_state,

    CAST(fs.order_date AS DATE) AS order_date

FROM gold.fact_sales fs

INNER JOIN gold.dim_partner dc
    ON fs.customer_id = dc.partner_id

GROUP BY
    fs.order_id,
    dc.partner_name,
    fs.order_state,
    fs.order_date;

In [0]:
SELECT *
FROM smart_odoo.gold.vw_monthly_sales_mom
ORDER BY month DESC LIMIT 1;

---------------------------------------------
Select * 
FROM smart_odoo.gold.vw_monthly_orders_mom
ORDER BY month DESC LIMIT 1;
---------------------------------------------
Select * 
FROM smart_odoo.gold.vw_monthly_aov_mom
ORDER BY month DESC LIMIT 1;
---------------------------------------------
Select *
FROM smart_odoo.gold.vw_monthly_customer_kpis
ORDER BY month DESC LIMIT 1;
---------------------------------------------
Select *
from smart_odoo.gold.vw_sales_revenue_trend
Order by month_number Asc;
---------------------------------------------
SELECT *
FROM gold.vw_top_products_sales
ORDER BY total_sales DESC
LIMIT 5;
---------------------------------------------
Select * 
FROM smart_odoo.gold.vw_recent_orders
ORDER BY order_date DESC
LIMIT 5;

